In [1]:
from datasets import load_dataset
from data_structures import Example
from hierarchical_optimizer import HierarchicalOptimizer
import config

CGEN_TRAIN_NUM = 40
CGEN_VAL_NUM   = 40
CGEN_TEST_NUM  = 40

GENERATION_METRICS = [
    {"name": "concept_coverage",    "weight": 0.25, "stage": 1},
    {"name": "spice",               "weight": 0.25, "stage": 1},
    {"name": "rouge_l",             "weight": 0.15, "stage": 1},
    {"name": "fluency",             "weight": 0.15, "stage": 3},
    {"name": "composition_quality", "weight": 0.15, "stage": 3},
    {"name": "diversity",           "weight": 0.05, "stage": 3},
]
config.METRICS_CONFIG = GENERATION_METRICS
config.METRIC_WEIGHTS = {m["name"]: m["weight"] for m in GENERATION_METRICS}

# ─── ЛОКАЛЬНАЯ ОПТИМИЗАЦИЯ ─────────────────────────────────────
config.LOCAL_ITERATIONS_PER_GENERATION = 3
config.LOCAL_CANDIDATES_PER_ITERATION  = 4
config.LOCAL_BATCH_SIZE                = 12
config.LOCAL_PARENTS_PER_ITERATION     = 3
config.MAX_GRADIENT_PAIRS              = 3
config.MINI_BATCH_RATIO                = 0.5
config.PRE_SCREEN_TOP_K                = 4
config.TRAIN_FAILURE_SAMPLE_SIZE       = 40

# ─── ГЛОБАЛЬНАЯ ОПТИМИЗАЦИЯ ────────────────────────────────────
config.GLOBAL_CANDIDATES              = 5 
config.GLOBAL_TRIGGER_INTERVAL        = 1
config.GLOBAL_QUALITY_GATE_TOLERANCE  = 0.75
config.GLOBAL_PRESCREEN_GATE          = 0.65
config.CROSSOVER_CANDIDATES           = 3
config.EXEMPLAR_COUNT                 = 5
config.FEW_SHOT_COUNT                 = 5
config.HISTORY_SCORE_THRESHOLD        = 0.25
config.GLOBAL_HISTORY_WINDOW          = 30

# ─── ПОПУЛЯЦИЯ И РАННЯЯ ОСТАНОВКА ──────────────────────────────
config.MAX_GENERATIONS                = 5
config.POPULATION_SIZE                = 3
config.PATIENCE                       = 2
config.FORCE_GLOBAL_AFTER_STAGNATION  = 1
config.MIN_IMPROVEMENT                = 0.003
config.SIMILARITY_THRESHOLD           = 0.75
config.DIVERSITY_WEIGHT               = 0.12

# ─── LLM ───────────────────────────────────────────────────────
config.TEMPERATURE                    = 0.4
config.MAX_TOKENS                     = 3000

# ─── ОЦЕНКА ────────────────────────────────────────────────────
config.MAX_EXAMPLES_PER_NODE          = 100
config.BATCH_EVAL_SIZE                = 40
config.JUDGE_BATCH_SIZE               = 20
config.CORRECTNESS_TOKEN_F1_THRESHOLD = 0.30
config.USE_CONTAINMENT_CHECK          = True
config.USE_LLM_CORRECTNESS_CHECK      = False
config.STABILITY_LAMBDA               = 0.05
config.BOOTSTRAP_RUNS                 = 3

# ─── СТАГНАЦИЯ И РАЗНООБРАЗИЕ ──────────────────────────────────
config.LOW_DIVERSITY_THRESHOLD        = 0.25
config.STAGNATION_SIMILARITY_THRESHOLD = 0.60
config.DIVERSITY_DISTANCE_THRESHOLD   = 0.35

print("Common Gen config applied")
print(f"   Metrics: {[m['name'] for m in GENERATION_METRICS]}")
print(f"   Weights: {config.METRIC_WEIGHTS}")
print(f"   Data splits: train={CGEN_TRAIN_NUM}, val={CGEN_VAL_NUM}, test={CGEN_TEST_NUM}")
print(f"   Generations: {config.MAX_GENERATIONS}, Population: {config.POPULATION_SIZE}")
print(f"   Local iters: {config.LOCAL_ITERATIONS_PER_GENERATION}, Beam: {config.LOCAL_PARENTS_PER_ITERATION}")
print(f"   Global candidates: {config.GLOBAL_CANDIDATES}, Trigger: every {config.GLOBAL_TRIGGER_INTERVAL} gen")
print(f"   Global gate: {config.GLOBAL_QUALITY_GATE_TOLERANCE}, Global prescreen: {config.GLOBAL_PRESCREEN_GATE}")
print(f"   Temperature: {config.TEMPERATURE}, Diversity weight: {config.DIVERSITY_WEIGHT}")


/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/kateav

Common Gen config applied
   Metrics: ['concept_coverage', 'spice', 'rouge_l', 'fluency', 'composition_quality', 'diversity']
   Weights: {'concept_coverage': 0.25, 'spice': 0.25, 'rouge_l': 0.15, 'fluency': 0.15, 'composition_quality': 0.15, 'diversity': 0.05}
   Data splits: train=40, val=40, test=40
   Generations: 5, Population: 3
   Local iters: 3, Beam: 3
   Global candidates: 5, Trigger: every 1 gen
   Global gate: 0.75, Global prescreen: 0.65
   Temperature: 0.4, Diversity weight: 0.12


In [2]:
from collections import defaultdict

def common_gen_to_examples(data):
    concept_groups = defaultdict(lambda: {"concepts": None, "targets": []})
    for item in data:
        idx = item["concept_set_idx"]
        concept_groups[idx]["concepts"] = item["concepts"]
        concept_groups[idx]["targets"].append(item["target"])

    examples = []
    for idx, group in concept_groups.items():
        concepts = group["concepts"]
        targets = group["targets"]
        input_text = f"Concepts: {', '.join(concepts)}"
        expected_output = targets[0]
        examples.append(Example(
            input_text=input_text,
            expected_output=expected_output,
            metadata={
                "concepts": concepts,
                "references": targets,
                "concept_set_idx": idx,
            },
        ))
    return examples


def get_common_gen_data(train_num: int, val_num: int, test_num: int):
    ds_train = load_dataset('allenai/common_gen', split='train')
    ds_val   = load_dataset('allenai/common_gen', split='validation')
    ds_test  = load_dataset('allenai/common_gen', split='test')

    # ds_train = ds_train.shuffle()
    # ds_val = ds_val.shuffle()
    # ds_test = ds_test.shuffle()

    train_examples = common_gen_to_examples(ds_train)
    validation_examples = common_gen_to_examples(ds_val)
    test_examples = common_gen_to_examples(ds_test)
    
    train_examples = train_examples[:train_num]
    validation_examples = validation_examples[:val_num]
    test_examples = test_examples[:test_num]

    return train_examples, validation_examples, test_examples


def data_fabric(dataset: str = 'common_gen', train_num: int = CGEN_TRAIN_NUM, val_num: int = CGEN_VAL_NUM, test_num: int = CGEN_TEST_NUM):
    common_gen_initial_prompt = (
        "Write a single natural English sentence that uses ALL of the given concepts. "
        "The sentence should describe a realistic everyday scenario, be grammatically correct, "
        "and use each concept naturally in context."
    )

    train_examples, validation_examples, test_examples = get_common_gen_data(
        train_num, val_num, test_num
    )
    return train_examples, validation_examples, test_examples, common_gen_initial_prompt

In [3]:
train_examples, validation_examples, test_examples, initial_prompt = data_fabric(
    'common_gen',
    train_num=CGEN_TRAIN_NUM,
    val_num=CGEN_VAL_NUM,
    test_num=CGEN_TEST_NUM,
)

print("Dataset prepared:")
print(f"  Train: {len(train_examples)} examples")
print(f"  Validation: {len(validation_examples)} examples")
print(f"  Test: {len(test_examples)} examples")
print()
print("Sample examples:")
for i, ex in enumerate(train_examples[:3], 1):
    print(f"  {i}. Input:    {ex.input_text}")
    print(f"     Expected: {ex.expected_output}")
    print(f"     Concepts: {ex.metadata['concepts']}")
    print(f"     Refs:     {len(ex.metadata['references'])} reference(s)")
    print()

Dataset prepared:
  Train: 40 examples
  Validation: 40 examples
  Test: 40 examples

Sample examples:
  1. Input:    Concepts: ski, mountain, skier
     Expected: Skier skis down the mountain
     Concepts: ['ski', 'mountain', 'skier']
     Refs:     3 reference(s)

  2. Input:    Concepts: wag, tail, dog
     Expected: The dog is wagging his tail.
     Concepts: ['wag', 'tail', 'dog']
     Refs:     3 reference(s)

  3. Input:    Concepts: lake, paddle, canoe
     Expected: woman paddling canoe on a lake
     Concepts: ['lake', 'paddle', 'canoe']
     Refs:     3 reference(s)



In [4]:
print("Initial prompt:")
print("-" * 60)
print(initial_prompt)
print("-" * 60)

Initial prompt:
------------------------------------------------------------
Write a single natural English sentence that uses ALL of the given concepts. The sentence should describe a realistic everyday scenario, be grammatically correct, and use each concept naturally in context.
------------------------------------------------------------


In [5]:
TASK_DESCRIPTION = (
    "Compositional generative task: given a set of everyday concepts (nouns/verbs), "
    "generate a single coherent, grammatically correct English sentence that naturally uses ALL "
    "of the provided concepts. The sentence should be plausible, fluent, and cover every concept."
)

optimizer = HierarchicalOptimizer(
    metrics_config=GENERATION_METRICS,
    task_description=TASK_DESCRIPTION,
)

In [6]:
best_node = optimizer.optimize(
    initial_prompt=initial_prompt,
    train_examples=train_examples,
    validation_examples=validation_examples,
    test_examples=test_examples,
    save_dir="./optimization_results_common_gen",
)

Evaluating initial prompt...
[diag] initial node: prompt_id=8bf17c8b163b len=205 chars
[diag] Prompt text: Write a single natural English sentence that uses ALL of the given concepts. The sentence should describe a realistic everyday scenario, be grammatically correct, and use each concept naturally in context.
[diag] evaluate_node: node_id=90708da1-af9c-4281-86dc-a611b8d3932a gen=0 source=initial split=validation stage=1 examples=40 seed_offset=0
[diag] evaluate_prompt: execute=True sample=True stage=1 incoming=40 will_use=40 weights={concept_coverage:0.25, rouge_l:0.15, spice:0.25} llm_calls_at_start=0
[diag] execute_prompt_batch: total=40 batch_size=50 n_batches=1 llm_calls_before=0


KeyboardInterrupt: 

In [ ]:
report = optimizer.get_optimization_report()
print('Optimization generations summary:')
for entry in report['optimization_log']:
    print(f"  Generation {entry['generation']}: time {entry['time']:.2f}s, best_score {entry['best_score']:.3f}")

local_stats = report['component_statistics']['local_optimizer']
print('Local optimizer summary:')
print(f"  Total iterations recorded: {local_stats.get('total_iterations', 0)}")
avg_it = local_stats.get('avg_iteration_time')
if avg_it is not None:
    print(f"  Avg iteration time: {avg_it:.2f}s")
else:
    print('  Avg iteration time: N/A')
print(f"  Total LLM calls attributed to local iterations: {local_stats.get('total_llm_calls_by_local', 0)}")
print('Per-iteration breakdown:')
for s in local_stats.get('iteration_stats', []):
    print(f"  Iter {s['iteration']}: time {s['time']:.2f}s, llm_calls {s['llm_calls']}")

Optimization generations summary:
  Generation 1: time 202.32s, best_score 0.523
  Generation 2: time 351.03s, best_score 0.536
  Generation 3: time 69.33s, best_score 0.554
  Generation 4: time 305.87s, best_score 0.560
  Generation 5: time 95.84s, best_score 0.560
Local optimizer summary:
  Total iterations recorded: 18
  Avg iteration time: 45.97s
  Total LLM calls attributed to local iterations: 276
Per-iteration breakdown:
  Iter 1: time 97.79s, llm_calls 54
  Iter 2: time 103.32s, llm_calls 25
  Iter 1: time 92.01s, llm_calls 54
  Iter 2: time 91.47s, llm_calls 20
  Iter 1: time 35.25s, llm_calls 6
  Iter 2: time 0.00s, llm_calls 0
  Iter 1: time 35.74s, llm_calls 6
  Iter 2: time 0.00s, llm_calls 0
  Iter 1: time 28.78s, llm_calls 5
  Iter 2: time 39.21s, llm_calls 9
  Iter 1: time 24.27s, llm_calls 5
  Iter 2: time 0.00s, llm_calls 0
  Iter 1: time 63.26s, llm_calls 46
  Iter 2: time 0.00s, llm_calls 0
  Iter 1: time 33.69s, llm_calls 6
  Iter 2: time 88.06s, llm_calls 20
  Ite

In [ ]:
print("BEST PROMPT FOUND:")
print("=" * 80)
print(best_node.prompt_text)
print("=" * 80)
print(f"Score: {best_node.metrics.composite_score():.3f}")
print(f"Generation: {best_node.generation}")
print(f"Source: {best_node.source.value}")

BEST PROMPT FOUND:
Craft a grammatically correct English sentence incorporating all of the given everyday concepts (nouns/verbs) in different sentence structures: [concept], [concept], [concept], [concept]. Your sentence should flow naturally, be contextually relevant, and logically tie together all concepts in a coherent and plausible manner.
Score: 0.560
Generation: 5
Source: local


In [ ]:
print("BASELINE vs OPTIMIZED — Test Set (Stage 3)")
print("=" * 80)

comparison = optimizer.compare_with_baseline(initial_prompt, test_examples)

print()
print("Summary:")
print(f"  {'Metric':<22s}  {'Baseline':>10s}  {'Optimized':>10s}  {'Delta':>10s}")
print(f"  {'-'*22}  {'-'*10}  {'-'*10}  {'-'*10}")
for name in ["concept_coverage", "spice", "rouge_l", "fluency", "composition_quality", "diversity"]:
    b = comparison["baseline"].get(name, 0.0)
    o = comparison["optimized"].get(name, 0.0)
    d = comparison["improvements"].get(name, 0.0)
    print(f"  {name:<22s}  {b:10.3f}  {o:10.3f}  {d:+10.3f}")
b_c = comparison["baseline"]["composite_score"]
o_c = comparison["optimized"]["composite_score"]
print(f"  {'COMPOSITE':<22s}  {b_c:10.3f}  {o_c:10.3f}  {o_c - b_c:+10.3f}")

BASELINE vs OPTIMIZED — Test Set (Stage 3)

Comparing with baseline...
[diag] evaluate_prompt: execute=True sample=True stage=3 incoming=40 will_use=40 weights={composition_quality:0.15, concept_coverage:0.25, diversity:0.05, fluency:0.15, rouge_l:0.15, spice:0.25} llm_calls_at_start=340
[diag] execute_prompt_batch: total=40 batch_size=50 n_batches=1 llm_calls_before=340
[diag]   batch 1/1: OK (40/40 done, llm_calls=341)
[diag] execute_prompt_batch done: llm_calls_delta=1 total_calls=341
[diag] evaluate_prompt execution done: llm_calls_for_execution=1
[diag]   example[0] expected='<None>' actual='I fell off the boat while trying to sail on the water.'
[diag]   example[1] expected='<None>' actual='The dog jumped over the wave at the beach.'
[diag]   example[2] expected='<None>' actual='I stand in front of the building wearing my new outfit.'
[diag]   ... and 37 more examples
[diag]   metric 'concept_coverage': 0.9738 (stage=1 weight=0.25 time=0.00s llm_calls=341)
[diag]   metric 'spice'

In [ ]:
print(optimizer.visualize_optimization_trajectory())


OPTIMIZATION TRAJECTORY

Generation | Best Score | Overall Best | Improvement
------------------------------------------------------------
   1       | 0.523      | 0.523       | +0.019 ██████████████████████████
   2       | 0.536      | 0.536       | +0.013 ██████████████████████████
   3       | 0.554      | 0.554       | +0.018 ███████████████████████████
   4       | 0.560      | 0.560       | +0.006 ███████████████████████████
   5       | 0.560      | 0.560       | +0.000 ███████████████████████████




In [ ]:
report = optimizer.get_optimization_report()

print("OPTIMIZATION REPORT:")
print("Overall Statistics:")
print(f"   Total time: {report['optimization_info']['total_time_seconds']:.2f}s")
print(f"   Generations: {report['optimization_info']['generations']}")
print(f"   Total nodes explored: {report['component_statistics']['history']['total_nodes']}")

print("Component Statistics:")
print(f"   Local optimizer iterations: {report['component_statistics']['local_optimizer']['total_iterations']}")
print(f"   Local improvements: {report['component_statistics']['local_optimizer']['improvements_count']}")
print(f"   Global optimizer steps: {report['component_statistics']['global_optimizer']['total_global_steps']}")
print(f"   Successful global changes: {report['component_statistics']['global_optimizer']['successful_global_changes']}")

print("Best Global Strategies:")
for i, strategy in enumerate(report['best_global_strategies'][:3], 1):
    print(f"   {i}. Score {strategy['score']:.3f}")
    print(f"      {strategy['strategy']['description'][:70]}...")

OPTIMIZATION REPORT:
Overall Statistics:
   Total time: 1032.43s
   Generations: 5
   Total nodes explored: 52
Component Statistics:
   Local optimizer iterations: 18
   Local improvements: 5
   Global optimizer steps: 2
   Successful global changes: 0
Best Global Strategies:
   1. Score 0.531
      crossover...
   2. Score 0.525
      crossover...
   3. Score 0.525
      meta-optimizer...


In [ ]:
lineage = optimizer.history.get_lineage(best_node.id)

print("EVOLUTION OF BEST PROMPT:")
print("=" * 80)

for i, node in enumerate(lineage):
    print(f"Step {i}: Generation {node.generation}, Source: {node.source.value}")
    if node.is_evaluated:
        print(f"  Score: {node.metrics.composite_score():.3f}")

    if node.operations:
        print(f"  Operations:")
        for op in node.operations:
            print(f"    - {op.description}...")

    if i < len(lineage) - 1:
        print("  ↓")

EVOLUTION OF BEST PROMPT:
Step 0: Generation 0, Source: initial
  Score: 0.505
  ↓
Step 1: Generation 1, Source: local
  Score: 0.514
  Operations:
    - Added specific guidelines on how to incorporate all concepts in a sentence....
  ↓
Step 2: Generation 2, Source: local
  Score: 0.523
  Operations:
    - Encouraged the use of diverse sentence structures to accommodate all concepts....
  ↓
Step 3: Generation 3, Source: local
  Score: 0.536
  Operations:
    - Emphasized the logical connection between concepts to ensure factual accuracy and coherence in the generated sentences....
  ↓
Step 4: Generation 4, Source: global
  Score: 0.512
  Operations:
    - Crossover (gen 4): top-2 x top-3...
  ↓
Step 5: Generation 5, Source: local
  Score: 0.560
  Operations:
    - Introduced a constraint for using varied sentence structures to avoid awkward constructions....


In [ ]:
print("FINAL SUMMARY")
print("=" * 80)

history_stats = optimizer.history.get_statistics()
local_stats = optimizer.local_optimizer.get_statistics()
global_stats = optimizer.global_optimizer.get_statistics()

print("Overall Statistics:")
print(f"  Total nodes explored: {history_stats['total_nodes']}")
print(f"  Evaluations performed: {history_stats['evaluated_nodes']}")
print(f"  Generations completed: {history_stats['max_generation']}")
print(f"  Best score achieved: {history_stats['best_score']:.3f}")
print(f"  Average score: {history_stats['avg_score']:.3f}")

print("Local Optimization:")
print(f"  Total iterations: {local_stats['total_iterations']}")
print(f"  Improvements found: {local_stats['improvements_count']}")
print(f"  Success rate: {local_stats['improvement_rate']:.1%}")

print("Global Optimization:")
print(f"  Total global steps: {global_stats['total_global_steps']}")
print(f"  Candidates generated: {global_stats['total_candidates_generated']}")
print(f"  Successful changes: {global_stats['successful_global_changes']}")
print(f"  Success rate: {global_stats['success_rate']:.1%}")

print("\nOptimization complete!")
print(f"   Results saved to: ./optimization_results_common_gen/")

FINAL SUMMARY
Overall Statistics:
  Total nodes explored: 52
  Evaluations performed: 52
  Generations completed: 6
  Best score achieved: 0.560
  Average score: 0.512
Local Optimization:
  Total iterations: 18
  Improvements found: 5
  Success rate: 27.8%
Global Optimization:
  Total global steps: 2
  Candidates generated: 12
  Successful changes: 0
  Success rate: 0.0%

Optimization complete!
   Results saved to: ./optimization_results_common_gen/
